# Random Cluster Array Generator (Nazca + gdstk)

**Author:** Jason P. Beech  
**Date:** 2026-03  
**Affiliation:** Tegenfeldt Lab / Lund University    

This notebook generates an array of unique random clustered objects using **gdstk** for boolean geometry and **Nazca** for layout export.

## What the code does
- Each object is built from overlapping circles with randomly sampled radii.
- A new circle is only accepted if it increases the total union area.
- One such cluster forms one object.
- A fresh random object is generated for each array position.

## Main settings
At the top of the code cell, you can change:

- `EXPORT_MODE`
  - `"plot"` → preview only
  - `"gds"` → export GDS only
- `N_circles` → number of circles used per object
- `radius_mean`, `radius_std` → radius distribution parameters
- `min_radius`, `max_radius` → radius truncation limits
- `NX`, `NY` → array size in x and y
- `margin_um` → spacing between neighboring objects
- `seed` → integer for reproducible random generation, or `None`

## Geometry notes
- Circle radii are sampled from a truncated normal distribution.
- Each new circle is attached to an existing one and must overlap it without full containment.
- A candidate circle is accepted only if it increases the union area by more than `area_tol`.
- The object pitch is estimated from an initial sample object, then enlarged by `margin_um`.

## Output behavior
- `"plot"` → shows a Nazca preview
- `"gds"` → exports a GDSII file

## Automatic filename
The exported GDS filename is generated automatically from the main parameters and the current date.

It includes:

- array size: `NX x NY`
- number of circles per object: `N_circles`
- mean radius: `radius_mean`
- margin: `margin_um`
- date in `YYYYMMDD` format

Example:

```python
random_clusters_20x20_N20_R12um_margin80um_20260317.gds

In [1]:
import nazca as nd
import gdstk
import numpy as np
import math
import random
from datetime import datetime

# =============================================================================
# Random Cluster Array Generator (Nazca + gdstk)
#
# Author: Jason P. Beech
# Date: 2026-03
# Affiliation: Tegenfeldt Lab / Lund University
#
# Description:
# Generates an array of unique random clustered objects built from overlapping
# circles. Boolean operations are handled with gdstk, and final polygons are
# placed/exported with Nazca.
# =============================================================================


# =============================================================================
# Output control
# =============================================================================
# Choose one of:
#   "plot" -> preview only
#   "gds"  -> export GDS only
EXPORT_MODE = "gds"


# =============================================================================
# User parameters
# =============================================================================
# --- object (particle) generator ---
N_circles = 20         # how many circles per object
radius_mean = 12.0     # µm (mean radius for normal distribution)
radius_std = 2.0       # µm (std dev for normal distribution)
min_radius = 6.0       # µm (truncate small radii)
max_radius = 20.0      # µm (optional safety cap)
poly_res = 128         # circle resolution (higher -> smoother, heavier)
precision = 1e-3       # boolean precision for gdstk
area_tol = 1e-3        # µm^2 minimum added area to accept a new circle
max_attempts = 500     # attempts per circle to find a valid placement
seed = None            # set int for reproducible results, or None for random

# --- array ---
NX, NY = 20, 20        # array size (columns x rows)
margin_um = 80.0       # extra clearance between objects (both axes)
layer_out = 1          # output layer for final polygons


# =============================================================================
# Auto-generated filename
# =============================================================================

date_tag = datetime.now().strftime("%Y%m%d")

GDS_FILENAME = (
    f"random_clusters_"
    f"{NX}x{NY}_"
    f"N{N_circles}_"
    f"R{radius_mean:g}um_"
    f"margin{margin_um:g}um_"
    f"{date_tag}.gds"
)


# =============================================================================
# Random seed
# =============================================================================

if seed is not None:
    random.seed(seed)
    np.random.seed(seed)


# =============================================================================
# Helper functions
# =============================================================================

def sample_radius(mu, sigma, rmin, rmax, maxtries=1000):
    """
    Sample a radius from a truncated normal distribution in [rmin, rmax].
    """
    for _ in range(maxtries):
        r = np.random.normal(mu, sigma)
        if rmin <= r <= rmax:
            return float(r)

    # Fallback if the truncation is too strict
    return float(max(rmin, min(rmax, mu)))


def circle_gdstk(cx, cy, R, res=128, layer=layer_out):
    """
    Create a circle polygon in gdstk.

    The tolerance is chosen so a full circle has approximately `res` points.
    """
    return gdstk.ellipse((cx, cy), R, tolerance=2 * math.pi / res, layer=layer)


def union_area(shape):
    """
    Return the total area of a gdstk Polygon / PolygonSet / list of such objects.
    """
    if shape is None:
        return 0.0

    total = 0.0

    if isinstance(shape, (list, tuple)):
        for s in shape:
            total += union_area(s)
        return total

    if hasattr(shape, "area"):
        return float(shape.area())

    if hasattr(shape, "get_polygons"):
        for arr in shape.get_polygons():
            total += float(gdstk.Polygon(arr).area())

    return total


def polys_from_gdstk(obj):
    """
    Convert a gdstk Polygon / PolygonSet / list into a list of point lists.
    """
    if obj is None:
        return []

    out = []
    items = obj if isinstance(obj, (list, tuple)) else [obj]

    for it in items:
        if hasattr(it, "polygons"):  # PolygonSet
            for arr in it.polygons:
                out.append([tuple(map(float, p)) for p in arr])

        elif hasattr(it, "points"):  # Polygon
            out.append([tuple(map(float, p)) for p in it.points])

        elif hasattr(it, "get_polygons"):
            for arr in it.get_polygons():
                out.append([tuple(map(float, p)) for p in arr])

    return out


def bbox_from_points_list(polylists):
    """
    Compute an axis-aligned bounding box across multiple polygons.
    """
    xs, ys = [], []

    for pts in polylists:
        if not pts:
            continue
        x, y = zip(*pts)
        xs.extend(x)
        ys.extend(y)

    if not xs:
        return (0, 0, 0, 0)

    return (min(xs), min(ys), max(xs), max(ys))


def put_polygon_list_at(polys, dx, dy, layer=layer_out):
    """
    Place a list of polygons into Nazca with translation (dx, dy).
    """
    for pts in polys:
        if not pts:
            continue

        tpts = [(x + dx, y + dy) for (x, y) in pts]

        if tpts[0] != tpts[-1]:
            tpts = tpts + [tpts[0]]

        nd.Polygon(tpts, layer=layer).put(0, 0)


# =============================================================================
# Core object generator
# =============================================================================

def generate_random_overlap_object(
    N, mu, sigma, rmin, rmax, res=128, precision=1e-3, area_tol=1e-3, max_attempts=500
):
    """
    Build one random clustered object from overlapping circles.

    Rules:
    - The first circle is placed at the origin.
    - Each new circle is attached to an existing circle.
    - The new circle must overlap the anchor circle without full containment.
    - The placement is accepted only if the total union area increases by > area_tol.

    Returns
    -------
    pts_list : list
        List of polygon point lists describing the union geometry.
    bbox : tuple
        Bounding box as (xmin, ymin, xmax, ymax).
    """
    r0 = sample_radius(mu, sigma, rmin, rmax)
    placed = [((0.0, 0.0), r0)]
    union_shape = circle_gdstk(0.0, 0.0, r0, res=res)

    for _ in range(N - 1):
        accepted = False
        r = sample_radius(mu, sigma, rmin, rmax)

        for _attempt in range(max_attempts):
            (ax, ay), Ra = random.choice(placed)
            phi = random.random() * 2.0 * math.pi

            eps = max(0.05, 0.02 * (Ra + r))
            dmin = abs(Ra - r) + eps
            dmax = (Ra + r) - eps

            if dmin >= dmax:
                continue

            d = random.uniform(dmin, dmax)
            cx = ax + d * math.cos(phi)
            cy = ay + d * math.sin(phi)

            new_circle = circle_gdstk(cx, cy, r, res=res)
            trial = gdstk.boolean(union_shape, new_circle, "or", precision=precision, layer=layer_out)

            if union_area(trial) > union_area(union_shape) + area_tol:
                union_shape = trial
                placed.append(((cx, cy), r))
                accepted = True
                break

        if not accepted:
            # If no valid placement is found after many attempts,
            # continue with fewer than N circles in this object.
            pass

    pts_list = polys_from_gdstk(union_shape)
    bbox = bbox_from_points_list(pts_list)
    return pts_list, bbox


# =============================================================================
# Build array of unique objects
# =============================================================================

# Generate one sample object to estimate the size and choose the pitch.
sample_pts, (xmin, ymin, xmax, ymax) = generate_random_overlap_object(
    N_circles,
    radius_mean,
    radius_std,
    min_radius,
    max_radius,
    res=poly_res,
    precision=precision,
    area_tol=area_tol,
    max_attempts=max_attempts,
)

obj_w = xmax - xmin
obj_h = ymax - ymin

pitchX = obj_w + margin_um
pitchY = obj_h + margin_um

# Center the array around the origin.
x0 = -0.5 * (NX - 1) * pitchX
y0 = -0.5 * (NY - 1) * pitchY

print(f"Array size: {NX} x {NY}")
print(f"Circles per object: {N_circles}")
print(f"Sample object size: {obj_w:.2f} µm x {obj_h:.2f} µm")
print(f"Pitch: {pitchX:.2f} µm x {pitchY:.2f} µm")
print(f"Export mode: {EXPORT_MODE}")
print(f"GDS filename: {GDS_FILENAME}")

# Place the sample object at the first array position.
put_polygon_list_at(sample_pts, x0 + 0 * pitchX, y0 + 0 * pitchY, layer=layer_out)

# Fill the rest of the array with fresh random objects.
for j in range(NY):
    for i in range(NX):
        if i == 0 and j == 0:
            continue

        pts, _bbox = generate_random_overlap_object(
            N_circles,
            radius_mean,
            radius_std,
            min_radius,
            max_radius,
            res=poly_res,
            precision=precision,
            area_tol=area_tol,
            max_attempts=max_attempts,
        )
        put_polygon_list_at(pts, x0 + i * pitchX, y0 + j * pitchY, layer=layer_out)


# =============================================================================
# Export / preview
# =============================================================================

mode = EXPORT_MODE.lower()

if mode == "gds":
    nd.export_gds(filename=GDS_FILENAME)
    print("CSG backend: gdstk")
    print(f"GDS exported: {GDS_FILENAME}")

elif mode == "plot":
    nd.export_plt()
    print("CSG backend: gdstk")
    print("Plot exported.")

else:
    raise ValueError("EXPORT_MODE must be one of: 'plot' or 'gds'")

Array size: 20 x 20
Circles per object: 20
Sample object size: 77.47 µm x 103.27 µm
Pitch: 157.47 µm x 183.27 µm
Export mode: gds
GDS filename: random_clusters_20x20_N20_R12um_margin80um_20260317.gds
Starting layout export...
...gds generation
...Wrote file './random_clusters_20x20_N20_R12um_margin80um_20260317.gds'


CSG backend: gdstk
GDS exported: random_clusters_20x20_N20_R12um_margin80um_20260317.gds
